In [1]:
import os
import gc
import sys
import glob
import yaml
import shutil
import random
import subprocess
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

# 1. Install Ultralytics cleanly
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)

import cv2
import torch
from tqdm.auto import tqdm
from ultralytics import YOLO

INPUT_ROOT = "/kaggle/input"
READY_DIR = "/tmp/rdd2022_ready"
SAVE_DIR = "/kaggle/working/yolo_results"
RUN_NAME = "rdd2022_yolo11m_clahe_800"

# 2. Locate last.pt from the checkpoint dataset automatically
last_pt_matches = glob.glob(os.path.join(INPUT_ROOT, "**", "last.pt"), recursive=True)
assert len(last_pt_matches) > 0, "ERROR: last.pt not found in /kaggle/input! Make sure rdd2022-yolo11m-ckpt is attached."
LAST_CKPT_SRC = last_pt_matches[0]
CKPT_ROOT_DIR = os.path.dirname(LAST_CKPT_SRC)
print(f"Found checkpoint to resume from: {LAST_CKPT_SRC}")

# Copy checkpoint & history files into /kaggle/working so Ultralytics resumes cleanly
shutil.rmtree(SAVE_DIR, ignore_errors=True)
os.makedirs(os.path.join(SAVE_DIR, RUN_NAME, "weights"), exist_ok=True)
RESUMED_LAST_PT = os.path.join(SAVE_DIR, RUN_NAME, "weights", "last.pt")
shutil.copy2(LAST_CKPT_SRC, RESUMED_LAST_PT)

for fname in ["best.pt"]:
    for candidate in [os.path.join(CKPT_ROOT_DIR, fname), os.path.join(os.path.dirname(CKPT_ROOT_DIR), "weights", fname)]:
        if os.path.exists(candidate):
            shutil.copy2(candidate, os.path.join(SAVE_DIR, RUN_NAME, "weights", fname))
            break

for fname in ["results.csv", "args.yaml"]:
    for candidate in [os.path.join(CKPT_ROOT_DIR, fname), os.path.join(os.path.dirname(CKPT_ROOT_DIR), fname)]:
        if os.path.exists(candidate):
            shutil.copy2(candidate, os.path.join(SAVE_DIR, RUN_NAME, fname))
            break

# 3. Rebuild exact same /tmp/rdd2022_ready dataset (excluding checkpoint folder images!)
shutil.rmtree(READY_DIR, ignore_errors=True)
os.makedirs(READY_DIR, exist_ok=True)

print("=== STEP 1: Locating RDD2022 images & labels ===")
all_imgs = glob.glob(os.path.join(INPUT_ROOT, "**", "*.jpg"), recursive=True) + \
           glob.glob(os.path.join(INPUT_ROOT, "**", "*.png"), recursive=True)

paired = []
for img_p in all_imgs:
    if "rdd2022-yolo11m-ckpt" in img_p:
        continue
    lbl_p = img_p.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
    if not os.path.exists(lbl_p):
        lbl_p = img_p.rsplit(".", 1)[0] + ".txt"
    if os.path.exists(lbl_p):
        paired.append((img_p, lbl_p))

print(f"Found {len(paired)} labeled images.")
assert len(paired) > 0, "ERROR: RDD2022 dataset images not found in /kaggle/input!"

print("=== STEP 2: Rebuilding Class Balancing & CLAHE in /tmp ===")
random.seed(42)
random.shuffle(paired)
split_idx = int(len(paired) * 0.85)
splits = {"train": paired[:split_idx], "val": paired[split_idx:]}

for s in ["train", "val"]:
    os.makedirs(os.path.join(READY_DIR, s, "images"), exist_ok=True)
    os.makedirs(os.path.join(READY_DIR, s, "labels"), exist_ok=True)

class_counts = Counter()
for img_p, lbl_p in splits["train"]:
    with open(lbl_p) as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                class_counts[int(parts[0])] += 1

max_c = max(class_counts.values()) if class_counts else 1
train_expanded = []
for img_p, lbl_p in splits["train"]:
    with open(lbl_p) as f:
        cls_ids = [int(line.split()[0]) for line in f if line.strip()]
    repeat = 1
    if cls_ids:
        min_cls_count = min(class_counts[c] for c in cls_ids)
        if min_cls_count < max_c * 0.45:
            repeat = 2
    for _ in range(repeat):
        train_expanded.append((img_p, lbl_p))

def process_image(args):
    idx, split, img_p, lbl_p = args
    img = cv2.imread(img_p)
    if img is None:
        return
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    enhanced = cv2.cvtColor(cv2.merge((cl, a, b)), cv2.COLOR_LAB2BGR)

    base_name = f"{split}_{idx:06d}.jpg"
    cv2.imwrite(os.path.join(READY_DIR, split, "images", base_name), enhanced, [int(cv2.IMWRITE_JPEG_QUALITY), 95])
    shutil.copy2(lbl_p, os.path.join(READY_DIR, split, "labels", f"{split}_{idx:06d}.txt"))

tasks = [(i, "train", img_p, lbl_p) for i, (img_p, lbl_p) in enumerate(train_expanded)] + \
        [(i, "val", img_p, lbl_p) for i, (img_p, lbl_p) in enumerate(splits["val"])]

with ThreadPoolExecutor(max_workers=4) as ex:
    list(tqdm(ex.map(process_image, tasks), total=len(tasks), desc="Applying CLAHE"))

yaml_path = os.path.join(READY_DIR, "data.yaml")
cfg = {
    "path": READY_DIR,
    "train": "train/images",
    "val": "val/images",
    "nc": 4,
    "names": ["D00_Longitudinal_Crack", "D10_Transverse_Crack", "D20_Alligator_Crack", "D40_Pothole"]
}
with open(yaml_path, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

# 4. Resume training from Epoch ~41 up to 60
print("=== STEP 3: Resuming YOLO11m Training up to Epoch 60 ===")
num_gpus = torch.cuda.device_count()
assert num_gpus > 0, "ERROR: GPU is not enabled! Set Accelerator to GPU T4 x2 in Session Options."
device = [0, 1] if num_gpus >= 2 else 0

model = YOLO(RESUMED_LAST_PT)
model.train(resume=True, device=device)

# Clear memory completely before final validation
del model
gc.collect()
torch.cuda.empty_cache()

# 5. Final Validation with TTA + Per-Class Breakdown
print("=== STEP 4: Final Validation with TTA ===")
best_weights = os.path.join(SAVE_DIR, RUN_NAME, "weights", "best.pt")
assert os.path.exists(best_weights), "ERROR: best.pt was not found!"

try:
    best_model = YOLO(best_weights)
    metrics = best_model.val(data=yaml_path, imgsz=800, batch=8, device=0, augment=True)
    print(f"FINAL TTA RESULTS -> mAP50: {metrics.box.map50:.4f} | mAP50-95: {metrics.box.map:.4f}")
except Exception as e:
    print(f"TTA validation skipped due to memory ({e}), running standard validation...")
    best_model = YOLO(best_weights)
    metrics = best_model.val(data=yaml_path, imgsz=800, batch=8, device=0, augment=False)
    print(f"FINAL STANDARD RESULTS -> mAP50: {metrics.box.map50:.4f} | mAP50-95: {metrics.box.map:.4f}")

names = ["D00_Longitudinal", "D10_Transverse", "D20_Alligator", "D40_Pothole"]
for i, ap50 in enumerate(metrics.box.ap50):
    print(f"Class {names[i]}: mAP50 = {ap50:.4f}")

print("=== ALL DONE SUCCESSFULLY! ===")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Found checkpoint to resume from: /kaggle/input/datasets/aliaamohamed472005/rdd2022-yolo11m-ckpt/last.pt
=== STEP 1: Locating RDD2022 images & labels ===
Found 38385 labeled images.
=== STEP 2: Rebuilding Class Balancing & CLAHE in /tmp ===


Applying CLAHE:   0%|          | 0/51180 [00:00<?, ?it/s]

=== STEP 3: Resuming YOLO11m Training up to Epoch 60 ===
Ultralytics 8.4.212 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=8.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.6, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/tmp/rdd2022_ready/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      39/60      6.45G       1.51      1.436      1.404          2        800: 100% ━━━━━━━━━━━━ 2447/2447 2.7it/s 15:23
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 164/164 2.3it/s 1:13
                   all       5225       8139      0.705      0.633      0.685      0.386

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      40/60      6.45G      1.483      1.395      1.389          1        800: 100% ━━━━━━━━━━━━ 2447/2447 2.6it/s 15:24
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 164/164 2.3it/s 1:13
                   all       5225       8139      0.706      0.634      0.686      0.386

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      41/60      6.41G      1.471      1.374      1.377          6        800: 100% ━